In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authentication successful!")

In [ ]:
!gsutil cp gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario/validation/validation.tfrecord-00000-of-00150 data.tfrecord

import os
if os.path.exists("data.tfrecord"):
    size = os.path.getsize("data.tfrecord")
    print(f"SUCCESS: File downloaded! Size: {size / 1024 / 1024:.2f} MB")
else:
    raise FileNotFoundError("Download failed.")

In [ ]:
!pip install waymo-open-dataset-tf-2-12-0 --no-deps
!pip install "protobuf<=3.20.3"

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import uuid
import sys
import os
from google.colab import files

sys.path.append("/usr/local/lib/python3.10/dist-packages")
try:
    from waymo_open_dataset.protos import scenario_pb2
except ImportError:
    import importlib
    importlib.reload(sys.modules["google.protobuf"])
    from waymo_open_dataset.protos import scenario_pb2

# --- HOW MANY INTERACTIVE SCENARIOS TO RENDER ---
MAX_SCENARIOS = 5

# --- DATA CONVERTER ---
def scenario_to_dict(proto):
    map_points = []
    for feat in proto.map_features:
        poly = []
        if feat.HasField("lane"): poly = feat.lane.polyline
        elif feat.HasField("road_line"): poly = feat.road_line.polyline
        elif feat.HasField("road_edge"): poly = feat.road_edge.polyline
        elif feat.HasField("crosswalk"): poly = feat.crosswalk.polygon
        elif feat.HasField("speed_bump"): poly = feat.speed_bump.polygon
        for p in poly:
            map_points.append([p.x, p.y, p.z])
    if not map_points:
        map_points = [[0, 0, 0]]
    roadgraph_xyz = np.array(map_points, dtype=np.float32)
    num_agents = len(proto.tracks)
    past_x     = np.zeros((num_agents, 10), dtype=np.float32)
    past_y     = np.zeros((num_agents, 10), dtype=np.float32)
    past_valid = np.zeros((num_agents, 10), dtype=np.int64)
    curr_x     = np.zeros((num_agents, 1),  dtype=np.float32)
    curr_y     = np.zeros((num_agents, 1),  dtype=np.float32)
    curr_valid = np.zeros((num_agents, 1),  dtype=np.int64)
    fut_x      = np.zeros((num_agents, 80), dtype=np.float32)
    fut_y      = np.zeros((num_agents, 80), dtype=np.float32)
    fut_valid  = np.zeros((num_agents, 80), dtype=np.int64)
    for i, track in enumerate(proto.tracks):
        for t in range(10):
            if track.states[t].valid:
                past_x[i, t] = track.states[t].center_x
                past_y[i, t] = track.states[t].center_y
                past_valid[i, t] = 1
        if track.states[10].valid:
            curr_x[i, 0] = track.states[10].center_x
            curr_y[i, 0] = track.states[10].center_y
            curr_valid[i, 0] = 1
        for t in range(80):
            if track.states[11 + t].valid:
                fut_x[i, t] = track.states[11 + t].center_x
                fut_y[i, t] = track.states[11 + t].center_y
                fut_valid[i, t] = 1
    return {
        "roadgraph_samples/xyz": roadgraph_xyz,
        "state/past/x": past_x,    "state/past/y": past_y,    "state/past/valid": past_valid,
        "state/current/x": curr_x, "state/current/y": curr_y, "state/current/valid": curr_valid,
        "state/future/x": fut_x,   "state/future/y": fut_y,   "state/future/valid": fut_valid,
    }

# --- COLOR MAP ---
# Green  = IsInteractive (flagged by Waymo as a safety-relevant interaction)
# Gray   = normal background agents
def get_interactive_colormap(proto):
    interactive_ids = set(proto.objects_of_interest)
    colors = np.full((len(proto.tracks), 4), [0.6, 0.6, 0.6, 0.4])  # gray default
    for i, track in enumerate(proto.tracks):
        if track.id in interactive_ids:
            colors[i] = [0.0, 0.8, 0.2, 1.0]  # green = interactive
    return colors

# --- VIEWPORT ---
def get_viewport(all_states, mask):
    valid = all_states[mask > 0]
    if valid.shape[0] == 0:
        return 0, 0, 100
    cy = (np.max(valid[..., 1]) + np.min(valid[..., 1])) / 2
    cx = (np.max(valid[..., 0]) + np.min(valid[..., 0])) / 2
    w  = max(np.ptp(valid[..., 1]), np.ptp(valid[..., 0]))
    return cy, cx, w

# --- SINGLE FRAME ---
def visualize_one_step(states, mask, roadgraph, title, cy, cx, width, color_map, size_pixels=600):
    fig, ax = plt.subplots(1, 1, num=uuid.uuid4())
    dpi = 100
    fig.set_size_inches([size_pixels / dpi, size_pixels / dpi])
    fig.set_dpi(dpi)
    fig.set_facecolor("white")
    ax.set_facecolor("white")
    ax.axis("off")
    fig.set_tight_layout(True)
    if len(roadgraph) > 0:
        rg = roadgraph[:, :2].T
        ax.plot(rg[0], rg[1], "k.", alpha=0.3, ms=1)
    mb = mask > 0
    ax.scatter(states[:, 0][mb], states[:, 1][mb], marker="o", linewidths=3, color=color_map[mb])
    ax.set_title(title, fontsize=10)
    s = max(10, width * 1.2)
    ax.axis([cx - s / 2, cx + s / 2, cy - s / 2, cy + s / 2])
    ax.set_aspect("equal")
    fig.canvas.draw()
    data = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    w, h = fig.canvas.get_width_height()
    img = data.reshape((h, w, 4))[:, :, :3]
    plt.close(fig)
    return img

# --- GENERATE ALL FRAMES ---
def generate_frames(data, color_map):
    px, py  = data["state/past/x"],    data["state/past/y"]
    cx, cy  = data["state/current/x"], data["state/current/y"]
    fx, fy  = data["state/future/x"],  data["state/future/y"]
    pm, cm_, fm = data["state/past/valid"], data["state/current/valid"], data["state/future/valid"]
    all_x   = np.concatenate([px, cx, fx], axis=1)
    all_y   = np.concatenate([py, cy, fy], axis=1)
    all_mask = np.concatenate([pm, cm_, fm], axis=1)
    vcy, vcx, vw = get_viewport(np.stack([all_x, all_y], axis=-1), all_mask)
    rg = data["roadgraph_samples/xyz"]
    imgs = []
    for i in range(10):
        s = np.stack([px[:, i], py[:, i]], -1)
        imgs.append(visualize_one_step(s, pm[:, i], rg, f"History: -{(10 - i) * 0.1:.1f}s", vcy, vcx, vw, color_map))
    for i in range(80):
        s = np.stack([fx[:, i], fy[:, i]], -1)
        imgs.append(visualize_one_step(s, fm[:, i], rg, f"Future: +{(i + 1) * 0.1:.1f}s", vcy, vcx, vw, color_map))
    return imgs

# --- RENDER + DOWNLOAD ---
def render_and_download(proto, filename):
    print(f"  Rendering {filename}...")
    color_map = get_interactive_colormap(proto)
    frames = generate_frames(scenario_to_dict(proto), color_map)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")
    im = ax.imshow(frames[0])
    ani = animation.FuncAnimation(
        fig, lambda f: [im.set_data(f)] or [im], frames=frames, interval=100
    )
    ani.save(filename, writer="ffmpeg", fps=10)
    plt.close()
    files.download(filename)
    print(f"  Downloaded: {filename}")

# --- SCAN + RENDER ---
FILENAME = "data.tfrecord"

if not os.path.exists(FILENAME):
    print("Error: data.tfrecord not found. Run the download cell first.")
else:
    print(f"Scanning for scenarios with IsInteractive agents (collecting up to {MAX_SCENARIOS})...\n")
    dataset = tf.data.TFRecordDataset(FILENAME, compression_type="")
    collected = []

    for raw in dataset:
        proto = scenario_pb2.Scenario()
        proto.ParseFromString(raw.numpy())

        interactive_ids = set(proto.objects_of_interest)
        if len(interactive_ids) == 0:
            continue  # skip scenarios with no interactive agents

        collected.append(proto)
        print(f"  Found: {proto.scenario_id}  |  Interactive agent IDs: {interactive_ids}")

        if len(collected) >= MAX_SCENARIOS:
            break

    print(f"\nRendering {len(collected)} videos (green = IsInteractive, gray = normal)...\n")
    for idx, proto in enumerate(collected):
        interactive_ids = set(proto.objects_of_interest)
        filename = f"interactive_{idx + 1:02d}_{proto.scenario_id[:8]}.mp4"
        print(f"Scenario {idx + 1}: {proto.scenario_id}  |  agents {interactive_ids}")
        render_and_download(proto, filename)

    print("\nAll done.")